In [12]:
# IMPORT, CONFIG

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re

from scrapers import MatchScraper, H2HScraper
from scrapers.base_scraper import BaseScraper
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

rankings = pd.read_csv("data/all_rankings2025.csv")
rankings.date = pd.to_datetime(rankings.date)

TEST_EVENT_ID = "8292"  # ESL Pro League Season 20
TEST_MATCH_INDEX = 0    # Első meccs

print(f"🎯 Konfiguráció:")
print(f"   Event ID: {TEST_EVENT_ID}")
print(f"   Match index: {TEST_MATCH_INDEX}")


🎯 Konfiguráció:
   Event ID: 8292
   Match index: 0


In [2]:
# 3. TARGET MATCHES SCRAPING
print("\n" + "="*60)
print("1️⃣ TARGET MATCHES SCRAPING")
print("="*60)

try:
    scraper = MatchScraper(headless=False)
    target_matches = scraper.scrape_event_matches(TEST_EVENT_ID)
    
    if target_matches.empty:
        raise ValueError("❌ Nincs target match!")
    
    # ordinal ragok eltávolítása
    target_matches["date_clean"] = target_matches["date"].apply(lambda x: re.sub(r'(\d+)(st|nd|rd|th)', r'\1', x))

    # dátummá alakítás
    target_matches["date_parsed"] = pd.to_datetime(target_matches["date_clean"], format="%B %d %Y")

    
    print(f"✅ {len(target_matches)} meccs találva")
    display(target_matches.head(3))
    
except Exception as e:
    print(f"❌ Hiba a target matches scraping közben: {e}")


1️⃣ TARGET MATCHES SCRAPING
✅ 40 meccs találva


,match_id,event_id,date,team_home,team_away,score_home,score_away,map_type,rounds,link,date_clean,date_parsed
0,2380078,8292,March 16th 2025,MOUZ,Vitality,0,3,bo5,5,https://www.hltv.org/matches/2380078/mouz-vs-v...,March 16 2025,2025-03-16
1,2380076,8292,March 15th 2025,Vitality,The MongolZ,2,1,bo3,3,https://www.hltv.org/matches/2380076/vitality-...,March 15 2025,2025-03-15
2,2380077,8292,March 15th 2025,Spirit,MOUZ,1,2,bo3,3,https://www.hltv.org/matches/2380077/spirit-vs...,March 15 2025,2025-03-15


In [3]:
# 4. MATCH KIVÁLASZTÁSA
print("\n" + "="*60)
print("2️⃣ MATCH KIVÁLASZTÁSA")
print("="*60)

if TEST_MATCH_INDEX >= len(target_matches):
    TEST_MATCH_INDEX = 0
    print(f"⚠️ Index túl nagy, első meccset használom")

selected_match = target_matches.iloc[TEST_MATCH_INDEX]

print(f"🎯 Kiválasztott meccs (index={TEST_MATCH_INDEX}):")
print(f"  Match ID:   {selected_match['match_id']}")
print(f"  Date:       {selected_match['date_parsed']}")
print(f"  Teams:      {selected_match['team_home']} vs {selected_match['team_away']}")
print(f"  Score:      {selected_match['score_home']} - {selected_match['score_away']}")
print(f"  URL:        {selected_match['link']}")


2️⃣ MATCH KIVÁLASZTÁSA
🎯 Kiválasztott meccs (index=0):
  Match ID:   2380078
  Date:       2025-03-16 00:00:00
  Teams:      MOUZ vs Vitality
  Score:      0 - 3
  URL:        https://www.hltv.org/matches/2380078/mouz-vs-vitality-esl-pro-league-season-21


In [4]:
# 5. MATCH H2H SCRAPING
print("\n" + "="*60)
print("3️⃣ MATCH H2H SCRAPING")
print("="*60)

try:
    match_url = selected_match['link']
    print(f"🔍 Scraping: {match_url}")
    
    scraper = H2HScraper(headless=False)
    h2h_df = scraper.scrape_match_h2h(match_url)
    
    if h2h_df.empty:
        raise ValueError("❌ H2H scraping sikertelen!")
    
    match_h2h = h2h_df.iloc[0]
    
    print(f"✅ H2H data:")
    display(match_h2h)
    
except Exception as e:
    print(f"❌ Hiba a H2H scraping közben: {e}")


3️⃣ MATCH H2H SCRAPING
🔍 Scraping: https://www.hltv.org/matches/2380078/mouz-vs-vitality-esl-pro-league-season-21
✅ H2H data:


match_id                                                          2380078
home_team                                                            MOUZ
home_team_id                                                         4494
away_team                                                        Vitality
away_team_id                                                         9565
wins_home                                                               5
wins_away                                                               9
overtimes                                                               1
total_non_overtime                                                     14
home_win_rate                                                      0.3571
home_team_avg_rating                                                 0.87
home_team_std_rating                                               0.0827
home_team_avg_ADR                                                   68.46
home_team_std_ADR                     

In [5]:
# 6. TEAMS EXTRACTION
print("\n" + "="*60)
print("4️⃣ TEAMS EXTRACTION")
print("="*60)

teams = {
    'home': {
        'team_id': str(match_h2h.get('home_team_id', 'unknown')),
        'team_name': match_h2h['home_team']
    },
    'away': {
        'team_id': str(match_h2h.get('away_team_id', 'unknown')), 
        'team_name': match_h2h['away_team']
    }
}

print(f"✅ Teams extracted:")
print(f"  Home: {teams['home']['team_name']} (ID: {teams['home']['team_id']})")
print(f"  Away: {teams['away']['team_name']} (ID: {teams['away']['team_id']})")


4️⃣ TEAMS EXTRACTION
✅ Teams extracted:
  Home: MOUZ (ID: 4494)
  Away: Vitality (ID: 9565)


In [6]:
# 7. TEAM HISTORY SCRAPING HELPER
print("\n" + "="*60)
print("5️⃣ TEAM HISTORY SCRAPING HELPER")
print("="*60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from scrapers.base_scraper import BaseScraper

class TeamHistoryScraper(BaseScraper):
    def scrape_team_matches(self, team_id: str, max_matches: int = 20):
        """Team match history scraping"""
        url = f"https://www.hltv.org/results?team={team_id}"
        self._init_driver()
        self.driver.get(url)

        wait = WebDriverWait(self.driver, 20)
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "results-holder")))

        self._random_delay()

        matches = []
        all_sublists = self.driver.find_elements(By.CLASS_NAME, "results-sublist")
        print(f"🔍 Összesen {len(all_sublists)} results-sublist betöltve")

        for sublist in all_sublists:
            if len(matches) >= max_matches:
                break

            try:
                headline = sublist.find_element(By.CLASS_NAME, "standard-headline").text.strip()
                match_date = headline.replace("Results for", "").strip()
            except:
                match_date = None

            match_blocks = sublist.find_elements(By.CLASS_NAME, "result-con")
            for match in match_blocks:
                if len(matches) >= max_matches:
                    break

                try:
                    a_tag = match.find_element(By.TAG_NAME, "a")
                    match_url = a_tag.get_attribute("href")
                    match_id = match_url.split('/')[4]

                    table = a_tag.find_element(By.TAG_NAME, "table")
                    tds = table.find_elements(By.TAG_NAME, "td")

                    team1_name = tds[0].find_element(By.CLASS_NAME, "team").text.strip()
                    team2_name = tds[2].find_element(By.CLASS_NAME, "team").text.strip()

                    score_spans = tds[1].find_elements(By.TAG_NAME, "span")
                    score1 = int(score_spans[0].text.strip())
                    score2 = int(score_spans[1].text.strip())

                    team1_html = tds[0].get_attribute("innerHTML")
                    won = "team-won" in team1_html

                    try:
                        map_text = a_tag.find_element(By.CSS_SELECTOR, ".map-text").text.strip()
                    except:
                        map_text = "bo1"

                    opponent_name = team2_name if team1_name else team1_name

                    matches.append({
                        "team_id": team_id,
                        "match_id": match_id,
                        "match_date": match_date,
                        "opponent_name": opponent_name,
                        "result": "win" if won else "loss",
                        "score_for": score1,
                        "score_against": score2,
                        "map_type": map_text,
                        "link": match_url
                    })

                except Exception as e:
                    print(f"⚠️ Hiba egy meccs feldolgozásánál: {e}")
                    continue

        self.close()
        return pd.DataFrame(matches)

print("✅ TeamHistoryScraper helper kész")


5️⃣ TEAM HISTORY SCRAPING HELPER
✅ TeamHistoryScraper helper kész


In [7]:
# 8. TEAM HISTORIES SCRAPING

print("\n" + "="*60)
print("6️⃣ TEAM HISTORIES SCRAPING")
print("="*60)

team_histories = {}

for side in ['home', 'away']:
    team_id = teams[side]['team_id']
    team_name = teams[side]['team_name']
    
    print(f"\n📈 Scraping history: {team_name} (ID: {team_id})")
    
    try:
        scraper = TeamHistoryScraper(headless=False)
        history_df = scraper.scrape_team_matches(team_id, max_matches=100)
        
        if history_df.empty:
            print(f"⚠️ Nincs history adat: {team_name}")
            team_histories[side] = pd.DataFrame()
            continue

        # ordinal ragok eltávolítása
        history_df["date_clean"] = history_df["match_date"].apply(lambda x: re.sub(r'(\d+)(st|nd|rd|th)', r'\1', x))
        # dátummá alakítás
        history_df["date_parsed"] = pd.to_datetime(history_df["date_clean"], format="%B %d %Y")

        # csak a meccs dátuma előttiek (megelőző meccsek)
        history_df = history_df[history_df['date_parsed'] < selected_match['date_parsed']]       

        if len(history_df) < 3:
            print(f"⚠️ Nincs elég history adat: {team_name}")
            team_histories[side] = pd.DataFrame()
            continue

        team_histories[side] = history_df
        
        print(f"✅ {len(history_df)} meccs találva")

        print(f"\n📋 Megelőző 3 meccs:")
        display(history_df.head(3))
        
    except Exception as e:
        print(f"❌ Hiba a history scraping közben: {e}")


6️⃣ TEAM HISTORIES SCRAPING

📈 Scraping history: MOUZ (ID: 4494)
🔍 Összesen 94 results-sublist betöltve
✅ 49 meccs találva

📋 Megelőző 3 meccs:


,team_id,match_id,match_date,opponent_name,result,score_for,score_against,map_type,link,date_clean,date_parsed
51,4494,2380077,March 15th 2025,Spirit,win,2,1,bo3,https://www.hltv.org/matches/2380077/spirit-vs...,March 15 2025,2025-03-15
52,4494,2380074,March 13th 2025,G2,win,2,1,bo3,https://www.hltv.org/matches/2380074/mouz-vs-g...,March 13 2025,2025-03-13
53,4494,2380065,March 10th 2025,Liquid,win,2,0,bo3,https://www.hltv.org/matches/2380065/mouz-vs-l...,March 10 2025,2025-03-10



📈 Scraping history: Vitality (ID: 9565)
🔍 Összesen 96 results-sublist betöltve
✅ 49 meccs találva

📋 Megelőző 3 meccs:


,team_id,match_id,match_date,opponent_name,result,score_for,score_against,map_type,link,date_clean,date_parsed
51,9565,2380076,March 15th 2025,The MongolZ,win,2,1,bo3,https://www.hltv.org/matches/2380076/vitality-...,March 15 2025,2025-03-15
52,9565,2380072,March 14th 2025,Liquid,win,2,0,bo3,https://www.hltv.org/matches/2380072/vitality-...,March 14 2025,2025-03-14
53,9565,2380059,March 9th 2025,MOUZ,win,2,0,bo3,https://www.hltv.org/matches/2380059/vitality-...,March 9 2025,2025-03-09


In [9]:
# 9. HISTORICAL H2H SCRAPING

# Get rank data function
def get_rank_data(date, team_name):
    team_rankings = rankings[(rankings['date'] < date) & 
                            (rankings['team_name'] == team_name)].reset_index(drop=True)

    # Current (latest) rank & points
    rank = team_rankings.loc[0, 'rank']
    points = team_rankings.loc[0, 'points']
    # Latest rank & points change
    rank_change = team_rankings.loc[0, 'rank'] - team_rankings.loc[1, 'rank']
    points_change = team_rankings.loc[0, 'points'] - team_rankings.loc[1, 'points']

    print(f"{team_name}\nRank: #{rank} ({rank_change:+.0f})\nPoints: {points} ({points_change:+.0f})")
    
    return({'rank': rank,
            'rank_change': rank_change, 
            'points': points,
            'points_change': points_change}
            )
print("Rank data function initialized.")

print("\n" + "="*60)
print("7️⃣ HISTORICAL H2H SCRAPING")
print("="*60)

historical_h2h = {}
N_MATCHES = 3

for side in ['home', 'away']:
    team_name = teams[side]['team_name']
    team_id = teams[side]['team_id']
    print(f"\n🔍 {team_name} last {N_MATCHES} matches H2H scraping:")
    
    history = team_histories.get(side, pd.DataFrame())
    
    if history.empty:
        print(f"  ⚠️ Nincs history, skip")
        historical_h2h[side] = {
            'avg_rating': None,
            'avg_adr': None, 
            'avg_swing': None
        }
        continue
    
    date = selected_match['date_parsed']
    history = history[history['date_parsed'] < date].reset_index(drop=True)
    
    # Last N match URL-ek
    last_n_matches = history.head(N_MATCHES)
    
    h2h_total = []
    h2h_wins = []
    ratings = []
    rating_stds = []
    adrs = []
    adr_stds = []
    swings = []
    swing_stds = []
    maps_total = []
    maps_wins = []
    maps_picked = []
    maps_avg_score_diffs = []
    ranks_opp = []
    rank_diffs = []
    points_opp = []
    points_diff = []
    
    for idx, match in last_n_matches.iterrows():
        match_url = match['link']
        print(f"  [{idx+1}/{N_MATCHES}] Scraping: {match['date_parsed'].date()} vs {match['opponent_name']}\n{match_url}")

        try:
            scraper = H2HScraper(headless=False)
            hist_h2h_df = scraper.scrape_match_h2h(match_url)

            if hist_h2h_df.empty:
                raise ValueError("❌ H2H scraping sikertelen!")
            
            if str(hist_h2h_df['home_team_id'][0]) == str(team_id):
                h2h_total.append(hist_h2h_df['total_non_overtime'][0])
                h2h_wins.append(hist_h2h_df['wins_home'][0])
                ratings.append(hist_h2h_df['home_team_avg_rating'][0])
                rating_stds.append(hist_h2h_df['home_team_std_rating'][0])
                adrs.append(hist_h2h_df['home_team_avg_ADR'][0])
                adr_stds.append(hist_h2h_df['home_team_std_ADR'][0])
                swings.append(hist_h2h_df['home_team_avg_Swing'][0])
                swing_stds.append(hist_h2h_df['home_team_std_Swing'][0])
                maps_total.append(hist_h2h_df['maps_played'][0])
                maps_wins.append(hist_h2h_df['home_maps_won'][0])
                maps_picked.append(hist_h2h_df['home_maps_picked'][0])
                maps_avg_score_diffs.append(hist_h2h_df['map_avg_score_diff'][0])
                
            elif str(hist_h2h_df['away_team_id'][0]) == str(team_id):
                h2h_total.append(hist_h2h_df['total_non_overtime'][0])
                h2h_wins.append(hist_h2h_df['wins_away'][0])
                ratings.append(hist_h2h_df['away_team_avg_rating'][0])
                rating_stds.append(hist_h2h_df['away_team_std_rating'][0])
                adrs.append(hist_h2h_df['away_team_avg_ADR'][0])
                adr_stds.append(hist_h2h_df['away_team_std_ADR'][0])
                swings.append(hist_h2h_df['away_team_avg_Swing'][0])
                swing_stds.append(hist_h2h_df['away_team_std_Swing'][0])
                maps_total.append(hist_h2h_df['maps_played'][0])
                maps_wins.append(hist_h2h_df['away_maps_won'][0])
                maps_picked.append(hist_h2h_df['maps_played'][0] - hist_h2h_df['home_maps_picked'][0])
                maps_avg_score_diffs.append( - hist_h2h_df['map_avg_score_diff'][0])
            else:
                print("ERROR: team_id not in df")
                        
        except Exception as e:
            print(f"    ⚠️ H2H scraping hiba: {e}")
            continue
   
    # Átlagok számítása
    avg_h2h_winrate = np.sum(h2h_wins) / np.sum(h2h_total) if h2h_total else None
    avg_rating = np.mean(ratings) if ratings else None
    avg_rating_std = np.mean(rating_stds) if rating_stds else None
    avg_adr = np.mean(adrs) if adrs else None
    avg_adr_std = np.mean(adr_stds) if adr_stds else None
    avg_swing = np.mean(swings) if swings else None
    avg_swing_std = np.mean(swing_stds) if swing_stds else None
    avg_maps_total = np.mean(maps_total) if maps_total else None
    avg_map_winrate = np.sum(maps_wins) / np.sum(maps_total) if maps_wins else None
    avg_map_pickrate = np.sum(maps_picked) / np.sum(maps_total) if maps_picked else None
    avg_map_score_diff = np.mean(maps_avg_score_diffs) if maps_avg_score_diffs else None

    historical_h2h[side] = {
        'avg_h2h_winrate': avg_h2h_winrate,
        'avg_rating': avg_rating,
        'avg_rating_std': avg_rating_std,
        'avg_adr': avg_adr,
        'avg_adr_std': avg_adr_std,
        'avg_swing': avg_swing,
        'avg_swing_std': avg_swing_std,
        'avg_maps_total': avg_maps_total,
        'avg_map_winrate': avg_map_winrate,
        'avg_map_pickrate': avg_map_pickrate,
        'avg_map_score_diff': avg_map_score_diff ,
        'n_matches_scraped': len(ratings)
    }

print(f"\n📊 Historical H2H stats összegzés:")
for side in ['home', 'away']:
    stats = historical_h2h[side]
    print(f"  {teams[side]['team_name']}:")
    print(f"    Avg H2H winrate:  {stats['avg_h2h_winrate']:.3f}" if stats['avg_h2h_winrate'] else "    Avg rating:  N/A")
    print(f"    Avg rating:  {stats['avg_rating']:.3f}" if stats['avg_rating'] else "    Avg rating:  N/A")
    print(f"    Avg ADR:     {stats['avg_adr']:.1f}" if stats['avg_adr'] else "    Avg ADR:     N/A")
    print(f"    Avg Swing:   {stats['avg_swing']:.1f}" if stats['avg_swing'] else "    Avg Swing:   N/A")
    print(f"    Avg map winrate:   {stats['avg_map_winrate']:.1f}" if stats['avg_map_winrate'] else "    Avg map winrate:   N/A")
    print(f"    Avg map score diff:   {stats['avg_map_score_diff']:.1f}" if stats['avg_map_score_diff'] else "    Avg map score diff:   N/A")


7️⃣ HISTORICAL H2H SCRAPING

🔍 MOUZ last 3 matches H2H scraping:
  [1/3] Scraping: 2025-03-15 vs Spirit
https://www.hltv.org/matches/2380077/spirit-vs-mouz-esl-pro-league-season-21
  [2/3] Scraping: 2025-03-13 vs G2
https://www.hltv.org/matches/2380074/mouz-vs-g2-esl-pro-league-season-21
  [3/3] Scraping: 2025-03-10 vs Liquid
https://www.hltv.org/matches/2380065/mouz-vs-liquid-esl-pro-league-season-21

🔍 Vitality last 3 matches H2H scraping:
  [1/3] Scraping: 2025-03-15 vs The MongolZ
https://www.hltv.org/matches/2380076/vitality-vs-the-mongolz-esl-pro-league-season-21
  [2/3] Scraping: 2025-03-14 vs Liquid
https://www.hltv.org/matches/2380072/vitality-vs-liquid-esl-pro-league-season-21
  [3/3] Scraping: 2025-03-09 vs MOUZ
https://www.hltv.org/matches/2380059/vitality-vs-mouz-esl-pro-league-season-21

📊 Historical H2H stats összegzés:
  MOUZ:
    Avg H2H winrate:  0.474
    Avg rating:  1.147
    Avg ADR:     77.7
    Avg Swing:   1.1
    Avg map winrate:   0.8
    Avg map score diff:

In [10]:
# 10. ROLLING FEATURES SZÁMÍTÁSA
print("\n" + "="*60)
print("8️⃣ ROLLING FEATURES SZÁMÍTÁSA")
print("="*60)

ml_input_row = {}

for side in ['home', 'away']:
    team_name = teams[side]['team_name']
    print(f"\n📊 {team_name} rolling features:")
    
    history = team_histories.get(side, pd.DataFrame())
    
    if history.empty:
        print(f"  ⚠️ Nincs history adat")
        continue
    
    # Last 3 winrate
    last_3 = history.head(3)
    last_3_wr = (last_3['result'] == 'win').mean() if len(last_3) > 0 else None
    
    # Last 5 winrate  
    last_5 = history.head(5)
    last_5_wr = (last_5['result'] == 'win').mean() if len(last_5) > 0 else None
    
    # Avg scores
    last_3_avg_for = last_3['score_for'].mean() if len(last_3) > 0 else None
    last_3_avg_against = last_3['score_against'].mean() if len(last_3) > 0 else None
    
    # Current streak
    streak = 0
    if len(history) > 0:
        last_result = history.iloc[0]['result']
        for _, match in history.iterrows():
            if match['result'] == last_result:
                streak += 1
            else:
                break
        if last_result == 'loss':
            streak *= -1
    
    print(f"  Last 3 winrate:    {last_3_wr:.3f}" if last_3_wr else "  Last 3 winrate:    N/A")
    print(f"  Last 5 winrate:    {last_5_wr:.3f}" if last_5_wr else "  Last 5 winrate:    N/A")
    print(f"  Last 3 avg score:  {last_3_avg_for:.1f} - {last_3_avg_against:.1f}" if last_3_avg_for else "  Last 3 avg score:  N/A")
    print(f"  Current streak:    {streak:+d}")
    
    # Store
    ml_input_row[f'{side}_last_3_winrate'] = last_3_wr
    ml_input_row[f'{side}_last_5_winrate'] = last_5_wr
    ml_input_row[f'{side}_last_3_avg_score_for'] = last_3_avg_for
    ml_input_row[f'{side}_last_3_avg_score_against'] = last_3_avg_against
    ml_input_row[f'{side}_current_streak'] = streak

    for key in historical_h2h[side].keys():
        value = historical_h2h[side][key]
        ml_input_row[f'{side}_{key}'] = value
        print(f"  {side}_{key}: {value:.2f}")

# Difference features
if 'home_last_3_winrate' in ml_input_row and 'away_last_3_winrate' in ml_input_row:
    diff_last3_wr = ml_input_row['home_last_3_winrate'] - ml_input_row['away_last_3_winrate']
    ml_input_row['diff_last_3_winrate'] = diff_last3_wr
    print(f"\n📊 Difference features:")
    print(f"  Diff last 3 WR:    {diff_last3_wr:+.3f}")


8️⃣ ROLLING FEATURES SZÁMÍTÁSA

📊 MOUZ rolling features:
  Last 3 winrate:    1.000
  Last 5 winrate:    0.800
  Last 3 avg score:  2.0 - 0.7
  Current streak:    +3
  home_avg_h2h_winrate: 0.47
  home_avg_rating: 1.15
  home_avg_rating_std: 0.14
  home_avg_adr: 77.73
  home_avg_adr_std: 6.07
  home_avg_swing: 1.11
  home_avg_swing_std: 1.93
  home_avg_maps_total: 2.67
  home_avg_map_winrate: 0.75
  home_avg_map_pickrate: 0.50
  home_avg_map_score_diff: 2.78
  home_n_matches_scraped: 3.00

📊 Vitality rolling features:
  Last 3 winrate:    1.000
  Last 5 winrate:    1.000
  Last 3 avg score:  2.0 - 0.3
  Current streak:    +10
  away_avg_h2h_winrate: 0.74
  away_avg_rating: 1.20
  away_avg_rating_std: 0.24
  away_avg_adr: 79.66
  away_avg_adr_std: 14.28
  away_avg_swing: 1.80
  away_avg_swing_std: 2.71
  away_avg_maps_total: 2.33
  away_avg_map_winrate: 0.86
  away_avg_map_pickrate: 0.43
  away_avg_map_score_diff: 5.50
  away_n_matches_scraped: 3.00

📊 Difference features:
  Diff last 

In [22]:
# Get rank data function

def get_rank_data(date, team_name):
    team_rankings = rankings[(rankings['date'] < date) & 
                            (rankings['team_name'] == team_name)].reset_index(drop=True)

    # Current (latest) rank & points
    rank = team_rankings.loc[0, 'rank']
    points = team_rankings.loc[0, 'points']
    # Latest rank & points change
    rank_change = team_rankings.loc[0, 'rank'] - team_rankings.loc[1, 'rank']
    points_change = team_rankings.loc[0, 'points'] - team_rankings.loc[1, 'points']

    print(f"{team_name}\nRank: #{rank} ({rank_change:+.0f})\nPoints: {points} ({points_change:+.0f})")
    
    return({'rank': rank,
            'rank_change': rank_change, 
            'points': points,
            'points_change': points_change}
            )

Falcons
Rank: #9 (+0)
Points: 256 (-20)
{'rank': 9, 'rank_change': 0, 'points': 256, 'points_change': -20}


In [40]:
# 11. RANKINGS ÉS EGYÉB FEATURE-ÖK
print("\n" + "="*60)
print("9️⃣ RANKINGS ÉS EGYÉB FEATURE-ÖK")
print("="*60)

# Mock rankings
ml_input_row['home_current_rank'] = 5
ml_input_row['away_current_rank'] = 8  
ml_input_row['home_rank_change'] = -1  # Javult
ml_input_row['away_rank_change'] = 2   # Romlott

print(f"  Home rank: #{ml_input_row['home_current_rank']} (change: {ml_input_row['home_rank_change']:+d})")
print(f"  Away rank: #{ml_input_row['away_current_rank']} (change: {ml_input_row['away_rank_change']:+d})")

# Basic match info
ml_input_row['match_id'] = selected_match['match_id']
ml_input_row['event_id'] = TEST_EVENT_ID
ml_input_row['date'] = selected_match['date_parsed']

# Date features
ml_input_row['date_month'] = selected_match['date_parsed'].strftime("%m")
ml_input_row['date_day'] = int(selected_match['date_parsed'].strftime("%w")) + 1

# Teams
ml_input_row['team_home'] = teams['home']['team_name']
ml_input_row['team_away'] = teams['away']['team_name']

# H2H features
ml_input_row['H2H_winrate_team1'] = match_h2h['home_win_rate']
ml_input_row['H2H_games'] = match_h2h['wins_home'] + match_h2h['wins_away']

# Match info
ml_input_row['match_rounds'] = selected_match['rounds']

# Player stats
ml_input_row['avg_rating_top3_team1'] = match_h2h.get('home_team_avg_rating', 1.0)

# Historical H2H features
if 'home' in historical_h2h:
    ml_input_row['home_hist_avg_rating'] = historical_h2h['home'].get('avg_rating')
    ml_input_row['home_hist_avg_adr'] = historical_h2h['home'].get('avg_adr')
    ml_input_row['home_hist_avg_swing'] = historical_h2h['home'].get('avg_swing')

if 'away' in historical_h2h:
    ml_input_row['away_hist_avg_rating'] = historical_h2h['away'].get('avg_rating')
    ml_input_row['away_hist_avg_adr'] = historical_h2h['away'].get('avg_adr')
    ml_input_row['away_hist_avg_swing'] = historical_h2h['away'].get('avg_swing')

# Historical difference features
if (ml_input_row.get('home_hist_avg_rating') and 
    ml_input_row.get('away_hist_avg_rating')):
    ml_input_row['hist_diff_rating'] = (
        ml_input_row['home_hist_avg_rating'] - 
        ml_input_row['away_hist_avg_rating']
    )

# Label (score)
ml_input_row['score_home'] = selected_match['score_home']
ml_input_row['score_away'] = selected_match['score_away']
ml_input_row['label_home_win'] = 1 if selected_match['score_home'] > selected_match['score_away'] else 0

print("✅ Feature-ök összegyűjtve!")


9️⃣ RANKINGS ÉS EGYÉB FEATURE-ÖK
  Home rank: #5 (change: -1)
  Away rank: #8 (change: +2)
✅ Feature-ök összegyűjtve!


In [41]:
# 12. VÉGEREDMÉNY - ML INPUT ROW
print("\n" + "="*60)
print("🔚 VÉGEREDMÉNY - ML INPUT ROW")
print("="*60)

# DataFrame-ként megjelenítés
ml_df = pd.DataFrame([ml_input_row])

print("📊 DataFrame nézet:")
display(ml_df.T.style.set_caption("ML Input Row - Transposed"))

print("\n📋 Részletes feature-ök:")
for key, value in ml_input_row.items():
    if isinstance(value, float):
        print(f"  {key:35s} = {value:.4f}")
    else:
        print(f"  {key:35s} = {value}")


🔚 VÉGEREDMÉNY - ML INPUT ROW
📊 DataFrame nézet:


,0
home_last_3_winrate,1.000000
home_last_5_winrate,0.800000
home_last_3_avg_score_for,2.000000
home_last_3_avg_score_against,0.666667
home_current_streak,3
home_avg_h2h_winrate,0.473684
home_avg_rating,1.146667
home_avg_rating_std,0.139033
home_avg_adr,77.733333
home_avg_adr_std,6.066667



📋 Részletes feature-ök:
  home_last_3_winrate                 = 1.0000
  home_last_5_winrate                 = 0.8000
  home_last_3_avg_score_for           = 2.0000
  home_last_3_avg_score_against       = 0.6667
  home_current_streak                 = 3
  home_avg_h2h_winrate                = 0.4737
  home_avg_rating                     = 1.1467
  home_avg_rating_std                 = 0.1390
  home_avg_adr                        = 77.7333
  home_avg_adr_std                    = 6.0667
  home_avg_swing                      = 1.1100
  home_avg_swing_std                  = 1.9267
  home_n_matches_scraped              = 3
  away_last_3_winrate                 = 1.0000
  away_last_5_winrate                 = 1.0000
  away_last_3_avg_score_for           = 2.0000
  away_last_3_avg_score_against       = 0.3333
  away_current_streak                 = 10
  away_avg_h2h_winrate                = 0.7407
  away_avg_rating                     = 1.2013
  away_avg_rating_std                 = 0.2436
 

# Other

In [55]:
# Rankings history

# Rankings

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

def scrape_team_rankings(url):
    driver = webdriver.Chrome()
    driver.get(url)
    
    # Várj, amíg betölt a lista
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ranked-team.standard-box")))
    
    url_prev = driver.find_element(By.CLASS_NAME, "pagination-prev ").get_attribute("href")

    date_raw = driver.find_element(By.CLASS_NAME, "regional-ranking-header-text").text
    date = date_raw.split("ranking on ")[-1]
    # ordinal ragok eltávolítása
    date_clean = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date)
    # dátummá alakítás
    date_parsed = pd.to_datetime(date_clean, format="%B %d, %Y")

    rankings = []
    rank_divs = driver.find_elements(By.CSS_SELECTOR, ".ranked-team.standard-box")
    
    for rank_div in rank_divs:
        try:
            rank = rank_div.find_element(By.CLASS_NAME, "position").text
            team_name = rank_div.find_element(By.CLASS_NAME, "name").text
            points = rank_div.find_element(By.CLASS_NAME, "points").text.replace('(', '').replace(')', '').replace(" HLTV points", "").replace(" points", "")
            
            team_link = rank_div.find_element(By.TAG_NAME, "a").get_attribute("href")
            team_id = team_link.split('/')[-2]
            profile_link = rank_div.find_element(By.CLASS_NAME, "moreLink").get_attribute("href")
            
            rankings.append({
                'date': date_parsed,
                'rank': int(rank.replace('#', '')),
                'team_id': team_id,
                'team_name': team_name,
                'points': int(points),
                'profile_link': profile_link
            })
        except Exception as e:
            print(f"Hiba: {e}")
            continue
    
    driver.quit()
    return pd.DataFrame(rankings), url_prev

# Összes rankings leszedése
all_rankings = pd.DataFrame()
url = "https://www.hltv.org/ranking/teams/"
url_prev = ""
while url_prev != "https://www.hltv.org/ranking/teams/2024/december/30":
    rankings, url_prev = scrape_team_rankings(url)
    print(f"Scraping: {url}")
    all_rankings = pd.concat([all_rankings, rankings], ignore_index=True)
    display(rankings.sample(3))
    print(f"+{len(rankings)} (Total: {len(all_rankings)})")
    url = url_prev

Scraping: https://www.hltv.org/ranking/teams/


,date,rank,team_id,team_name,points,profile_link
225,2025-10-13,226,20899,Square Sausages,2,https://www.hltv.org/team/13434/square-sausages
25,2025-10-13,26,16551,Lynn Vision,63,https://www.hltv.org/team/8840/lynn-vision
220,2025-10-13,221,19998,Atrix,2,https://www.hltv.org/team/12533/atrix


+261 (Total: 261)
Scraping: https://www.hltv.org/ranking/teams/2025/october/6


,date,rank,team_id,team_name,points,profile_link
86,2025-10-06,87,16558,Kaleido,14,https://www.hltv.org/team/13331/kaleido
140,2025-10-06,141,7131,Alter Ego,7,https://www.hltv.org/team/9958/alter-ego
25,2025-10-06,26,7528,fnatic,86,https://www.hltv.org/team/4991/fnatic


+262 (Total: 523)
Scraping: https://www.hltv.org/ranking/teams/2025/september/29


...

+230 (Total: 9835)
Scraping: https://www.hltv.org/ranking/teams/2025/january/13


,date,rank,team_id,team_name,points,profile_link
166,2025-01-13,167,19133,Gods Reign,3,https://www.hltv.org/team/12415/gods-reign
103,2025-01-13,104,9217,Tropa do Taco,8,https://www.hltv.org/team/13025/tropa-do-taco
139,2025-01-13,140,22643,TNL,4,https://www.hltv.org/team/12833/tnl


+221 (Total: 10056)
Scraping: https://www.hltv.org/ranking/teams/2025/january/6


,date,rank,team_id,team_name,points,profile_link
75,2025-01-06,76,9254,KOI,13,https://www.hltv.org/team/12591/koi
211,2025-01-06,212,21648,SKYFURY,1,https://www.hltv.org/team/12982/skyfury
113,2025-01-06,114,20006,hypewrld,7,https://www.hltv.org/team/12570/hypewrld


+229 (Total: 10285)


In [59]:
# Save rankings history

all_rankings.to_csv("data/all_rankings2025.csv", index=False)